# DB head

> Inference DB head, and the nearest upsample the detection neck shares.


In [ ]:
#| default_exp layers.neck


#| export
Two stride-2 transposed convolutions restore the input resolution. The probability map is a sigmoid over one channel.


In [ ]:
#| hide
from nbdev import show_doc


In [ ]:
#| export
#| notest
from __future__ import annotations

from max.graph import TensorValue, ops
from max.nn.layer import Module

from solid_lens.layers.activations import relu
from solid_lens.layers.conv import ConvTranspose2x2, conv2d


In [ ]:
#| export
#| notest
def _upsample(x: TensorValue, factor: int) -> TensorValue:
    if factor == 1:
        return x
    shape = x.shape
    return ops.resize_nearest(
        x,
        [shape[0], shape[1], shape[2] * factor, shape[3] * factor],
        coordinate_transform_mode=2,
        round_mode=2,
    )


`coordinate_transform_mode=2` and `round_mode=2` are the MAX names for the nearest-neighbor mode the exported graph uses. RepLKPAN calls this with factors 2, 4, and 8.


In [ ]:
#| export
#| notest
class DBHead(Module):
    def __init__(self, in_channels: int, hidden: int) -> None:
        super().__init__()
        self.conv = conv2d(in_channels, hidden, 3, padding=1, bias=True)
        self.up1 = ConvTranspose2x2(hidden, hidden, bias=True)
        self.up2 = ConvTranspose2x2(hidden, 1, bias=True)

    def __call__(self, x: TensorValue) -> TensorValue:
        x = relu(self.conv(x))
        x = relu(self.up1(x))
        return ops.sigmoid(self.up2(x))


The 3×3 reduces the fused neck to 64 channels. Each transpose doubles the spatial size. The second one emits the single-channel probability map.
